# Lesson 3 — Streaming tokens as they arrive

**Goal:** print the reply piece-by-piece as it generates, instead of waiting
for the whole thing.

## One concept: `.stream()` instead of `.invoke()`

Every Runnable supports both:
- `.invoke(x)` → run fully, return the final result
- `.stream(x)` → a generator that yields pieces as they're produced

### The proxy parallel — `api/proxy/database.py:318-332`
Your `extract_tokens_from_chunk` splits raw SSE bytes on `\n`, strips the
`data: ` prefix, skips `[DONE]`, and `json.loads` each chunk to reach
`delta.content`. LangChain does all that parsing internally — you just loop
over decoded text.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(base_url='http://127.0.0.1:8080/v1', api_key='not-needed',
                   model='qwen3-14b-q4_k_m', temperature=0.7)
chain = ChatPromptTemplate.from_messages([('human', 'Write a short haiku about {topic}.')]) | model | StrOutputParser()

### Stream it
`end=''` + `flush=True` make each chunk appear live instead of buffering.

In [ ]:
for chunk in chain.stream({'topic': 'a compiler finally passing all tests'}):
    print(chunk, end='', flush=True)

### Accumulate while streaming — the proxy's pattern
Your `streamer()` fills a list so it can save the full reply to SQLite after
the stream ends. Same idea here: collect chunks, then join.

In [ ]:
collected = []
for chunk in chain.stream({'topic': 'a green test suite'}):
    print(chunk, end='', flush=True)
    collected.append(chunk)

full_reply = ''.join(collected)  # <- what you'd save to the DB
print('\n\n--- captured', len(full_reply), 'chars for saving ---')